In [1]:
from dotenv import load_dotenv
# input path to .env file, which should contain TOGETHER_API_KEY
load_dotenv(override=True)

%load_ext autoreload
%autoreload 2

import pickle
import os
import sys
from typing import List
from litellm import completion

# Add the webarena directory to the Python path so we can import browser_env
sys.path.append('../webarena')

# Import the necessary modules that the pickle file depends on
try:
    from browser_env import *
    from agent import *
    print("Successfully imported webarena modules")
except ImportError as e:
    print(f"Warning: Could not import some modules: {e}")
    print("This might affect pickle loading, but we'll try anyway...")
    
    # Try alternative import approach
    try:
        import webarena.browser_env
        import webarena.agent
        print("Successfully imported webarena modules with alternative approach")
    except ImportError as e2:
        print(f"Alternative import also failed: {e2}")
        print("We'll proceed with pickle loading anyway...")

Python executable: /Users/deanorenstein/Documents/academic/self improving AI agents/project/ARMPA/.venv/bin/python
Successfully imported webarena modules


/Users/deanorenstein/Documents/academic/self improving AI agents/project/ARMPA/.venv/lib/python3.10/site-packages/beartype/_util/hint/pep/utilpeptest.py:345: BeartypeDecorHintPep585DeprecationWarning: PEP 484 type hint typing.Mapping[str, gymnasium.spaces.space.Space[typing.Any]] deprecated by PEP 585 scheduled for removal in the first Python version released after October 5th, 2025. To resolve this, import this hint from "beartype.typing" rather than "typing". See this discussion for further details and alternatives:
    https://github.com/beartype/beartype#pep-585-deprecations
  warn(
/Users/deanorenstein/Documents/academic/self improving AI agents/project/ARMPA/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/deanorenstein/Documents/academic/self improving AI agents/project/ARMPA/.venv/li

## View the sample trajectory to get familiar

In [2]:
trajectory_data = pickle.load(open("../webarena/trajectory.pkl", "rb"))
#trajectory_data = pickle.load(open("../webarena/runs/20251114113616/trajectories/760.pkl", "rb"))
task_from_trajectory = "What is the top-1 best-selling brand in Quarter 1 2022"

In [3]:
import json

print(f"Displaying {len(trajectory_data)} trajectory items:")
print("=" * 80)

for i, item in enumerate(trajectory_data):
    print(f"\n--- Trajectory Item {i+1} ---")
    try:
        # Try to convert to JSON-serializable format
        if hasattr(item, '__dict__'):
            # If it's an object with attributes, convert to dict
            item_dict = item.__dict__
        elif isinstance(item, (dict, list, str, int, float, bool, type(None))):
            # If it's already JSON-serializable
            item_dict = item
        else:
            # Try to convert to string representation
            item_dict = str(item)
        
        # Pretty print as JSON
        print(json.dumps(item_dict, indent=2, default=str))
        
    except Exception as e:
        print(f"Could not serialize item {i+1}: {e}")
        print(f"Raw item: {item}")
    
    print("-" * 40)


Displaying 34 trajectory items:

--- Trajectory Item 1 ---
{
  "observation": {
    "text": "Tab 0 (current): Dashboard / Magento Admin\n\n[251] RootWebArea 'Dashboard / Magento Admin' focused: True url: http://ec2-3-128-25-196.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/\n\t[547] link 'Magento Admin Panel' url: http://ec2-3-128-25-196.us-east-2.compute.amazonaws.com:7780/admin/admin/\n\t\t[548] image 'Magento Admin Panel' url: http://ec2-3-128-25-196.us-east-2.compute.amazonaws.com:7780/static/version1681922233/adminhtml/Magento/backend/en_US/images/magento-icon.svg\n\t[550] menubar '' orientation: horizontal\n\t\t[552] link '\\ue604 DASHBOARD' url: http://ec2-3-128-25-196.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/\n\t\t[555] link '\\ue60b SALES' url: http://ec2-3-128-25-196.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/#\n\t\t[588] link '\\ue608 CATALOG' url: http://ec2-3-128-25-196.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboar

The entires alternate between observation and action. In observations, we can reference the top level `observation` field, and have an LLM summarize what it sees in a few sentences. In the actions, we can take the top level `llm_reasoning`, which is what our reasoning-acting LLM was thinking as it chose that action. By representing trajectories as pairs of `observation`-`llm_reasoning` in our architect agent system prompt, it can effectively compress that trjaectory into a reusable strategy.

## Test storing and retrieving memory in the QDrant DB from this trajectory

In [2]:
# Add the parent directory to Python path so we can import memory module
sys.path.append('..')

# Import MemoryManager
from memory.manager import MemoryManager

Python executable: /Users/deanorenstein/Documents/academic/self improving AI agents/project/ARMPA/.venv/bin/python


In [3]:
collection_name = "webarena"

In [4]:
mm = MemoryManager(collection_name=collection_name)

Collection 'webarena' already exists, using existing collection.


In [9]:
trajectory_success = False

In [16]:
action, resulting_observation = trajectory_data[1], trajectory_data[2]

In [ ]:
from webarena.browser_env import ActionTypes, get_action_description

In [52]:
action_str = get_action_description(
    action,
    resulting_observation["info"]["observation_metadata"],
    action_set_tag="id_accessibility_tree",
    prompt_constructor=agent.prompt_constructor
)

In [ ]:
summary = mm.summarize_webarena_observation(resulting_observation['observation']['text'])

In [78]:
mm.client.upsert(collection_name=mm.collection_name,points=[mm._get_trajectory_step_point(
    goal=task_from_trajectory,
    obs_text=summary,
    action_taken=action_str,
    success=trajectory_success,
    step_id=1
)])

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [55]:
mm.print_all_memories()

🧠 Found 10 memories in collection 'webarena':

--- Memory 1 (ID: 078dbde6-a5e2-42d4-96d3-be80bb886370) ---
📝 Goal: What is the top-1 best-selling brand in Quarter 1 2022
👁️  Observation: **Summary of Web Observation: Adobe Commerce Reports Menu**

The page titled **“Reports menu | Adobe Commerce”** on Adobe Experience League provides an overview of the reporting capabilities available in Adobe Commerce’s admin interface. It explains that the **Reports menu** offers centralized access to a comprehensive suite of reports covering key business areas: **sales, products, customers, marketing efforts, and promotions**.

Key details:
- **Purpose**: Helps administrators and users monitor business performance through structured, actionable insights.
- **Audience**: Designed for **Beginner, Intermediate, Admin, Leader, and User** roles.
- **Content Structure**: 
  - Includes a visual diagram of the Reports menu.
  - Lists all available report categories under the menu: **Marketing, Reviews, Sale

In [ ]:
mm.reset_collection('webarena')

🗑️  Deleted collection 'webarena'
✅ Recreated empty collection 'webarena'


In [ ]:
observations_actions_reasonings = mm.store_trajectory_testing( # Note: official method is named something else now - we summarize observations in run.py, and pass these with actions+reasonings into this method at end of trajectory
    trajectory=trajectory_data,
    goal=task_from_trajectory,
    success=trajectory_success,
    prompt_constructor=agent.prompt_constructor)
# 32 sec for storing 17 points in the DB

In [15]:
import pickle
with open("observations_actions_reasonings.pkl", "wb") as f:
    pickle.dump(observations_actions_reasonings, f)

In [ ]:
print(observations_actions_reasonings)
# We would send something like this through the LLM which gives per-step reward for the memories' influence at each step

In [ ]:
obs = trajectory_data[2]
mm.cue_based_recall(summarized_obs=summary,
        goal=task_from_trajectory,
        top_k=10)

[{'score': 0.959414,
  'memory_id': '07f40dae-674d-4ab7-8768-2419cb73c8b0',
  'step_id': 1,
  'goal': 'What is the top-1 best-selling brand in Quarter 1 2022',
  'obs_summary': '**Summary of Web Observation – Magento Admin Dashboard**\n\nThe user is currently on the **Magento Admin Dashboard** at `http://ec2-3-128-25-196.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/`. The interface is fully loaded with administrative navigation and reporting tools.\n\n### Key Observations:\n\n1. **Navigation Menu**:\n   - The top horizontal menu includes all standard Magento admin sections: **Dashboard, Sales, Catalog, Customers, Marketing, Content, Reports, Stores, System**, and **Find Partners & Extensions**.\n   - The **Reports** menu is expanded, revealing detailed sub-sections for **Marketing, Reviews, Sales, Customers, Products, Statistics, and Business Intelligence**, with links to specific reports (e.g., Abandoned Carts, Bestsellers, Tax, Orders, Low Stock, etc.).\n\n2. **System A

## LLM-as-judge reward

Getting familiar with how we would get reward from a trajectory. We pass in a list of (<summarized observation>, <reason for next action>) tuples, and have the judge give us per-trajectory-step reward scores in [0, 1].

In [32]:
from memory.prompts.reward import per_step_llm_judge_prompt
observations_actions_reasonings = pickle.load(open("../webarena/runs/20251030175025/trajectories/1.pkl", "rb"))

In [36]:
def clean_reasoning(reason):
    return reason.replace('`', '').replace('\n', '').strip()

formatted_steps = "\n\n---\n\n".join(
    [
        f"""### Step {i+1}

**Observation:**  
{obs.replace('`', '').strip()}

**Reasoning:**  
{clean_reasoning(reason)}"""
        for i, (obs, action, reason) in enumerate(observations_actions_reasonings)
    ]
)

prompt = per_step_llm_judge_prompt.format(
    task="What is the top-1 best-selling brand in Quarter 1 2022",
    observations_actions_reasonings=formatted_steps,
    success=False
)

In [ ]:
print(prompt)

In [44]:
response_data = generate_from_litellm_completion(
    prompt=prompt,
    model="together_ai/OpenAI/gpt-oss-120B",
    temperature=0.7,
)

In [47]:
print(response_data['choices'][0]['message']['content'], type(response_data['choices'][0]['message']['content']))

{
  "step_scores": [
    {
      "step": 1,
      "reasoning": "Correctly identified that the dashboard lacks brand and time-filtered data, and correctly proposed navigating to the Reports section as the next logical step.",
      "score": 1.0
    },
    {
      "step": 2,
      "reasoning": "Accurately identified Advanced Reporting as a promising path for date-brand filtering, though it overestimated its likelihood of showing brand data without confirmation.",
      "score": 0.8
    },
    {
      "step": 3,
      "reasoning": "Correctly redirected focus to the more standard 'Bestsellers' report under Sales, which is more likely to have date filters and product data than Advanced Reporting.",
      "score": 1.0
    },
    {
      "step": 4,
      "reasoning": "Correctly identified the need to set date filters for Q1 2022 and planned the correct inputs for 'From' and 'To' fields.",
      "score": 1.0
    },
    {
      "step": 5,
      "reasoning": "Reiterated correct date setup but fa

In [49]:
# conver to json
import json
response_data = json.loads(response_data['choices'][0]['message']['content'])

## Recall agent

This is the prompt for the retrieval agent in memory-R1:

```python
"""
You are an intelligent memory assistant
tasked with retrieving
accurate information from conversation memories.

# CONTEXT:
You have access to memories from two speakers in a conversation.
These memories contain timestamped information that may be relevant to answering the question.

# INSTRUCTIONS:
1. Carefully analyze all provided memories from both speakers
2. Pay special attention to the timestamps to determine the answer
3. If the question asks about a specific event or fact, look for direct evidence
4. If the memories contain contradictory information, prioritize the most recent memory
5. If there is a question about time references (like "last year"
, "two months ago"),
calculate the actual date based on the memory timestamp.
6. Always convert relative time references to specific dates, months, or years.
7. Focus only on the content of the memories. Do not confuse character names
8. The answer should be less than 5-6 words.
9. IMPORTANT: Select memories you found that are useful for answering the questions, and output it before you answer questions.
10. IMPORTANT: Output the final
answer after **Answer:**

# APPROACH (Think step by step):
1. Examine all relevant memories
2. Examine the timestamps carefully
3. Look for explicit mentions that answer the question
4. Convert relative references if needed
5. Formulate a concise answer
6. Double-check the answer correctness
7. Ensure the final
answer is specific
8. First output the memories that you found are important before you answer questions

Memories for user John:
- 7:20 pm on 16 June, 2023: John has a special memory of a vacation to California where he experienced a gorgeous sunset and an enjoyable night strolling the shore, creating meaningful memories with loved ones.
- 6:13 pm on 10 April, 2023: John explored the coast in the Pacific Northwest and visited some national parks, finding the beauty of nature absolutely breathtaking.
- 3:14 pm on 13 August, 2023: John enjoys spending time outdoors with his family, including activities such as hiking, hanging out at the park, and having picnics. He also values indoor family activities like playing board games and having movie nights at home.
... (In total 30 most relevant memories from John's Memory Bank are provided) ...

Memories for user Maria:
- 6:29 pm on 7 July, 2023: John experienced a severe flood in his old area last week, which caused significant damage to homes due to poor infrastructure.
- 1:24 pm on 25 May, 2023: Maria appreciates the beauty of small, meaningful moments in life, as reflected in her reaction to a family beach photo shared by John.
- 3:14 pm on 13 August, 2023: Maria appreciates family bonding and is interested in the activities that John and his family enjoy doing together.
... (In total 30 most relevant memories from Maria's Memory Bank are provided) ...

Question: Does John live close to a beach or the mountains?
"""
```

How it adapts to our problem:
- Instead of recall memories on a conversation turn, we recall them on a step in a reasoning-planning-acting trajectory for a complex, long horizon task
- Instead of facts about users, which are used to help the agent give the best, most accurate answer, we recall by semantically searching against environment cue embeddings, and then take its corresponding metadata as clues for what to do next, so as to maximize task success and minimize number of steps to get there
- Memories can look like these:
    - A ReasoningBank-style skill or learned heuristic so that later tasks can be navigated more effectively
    - Cue-action mappings formatted like this: **"Last time I was in a similar situation, I tried doing the corresponding action, and it led to <success/failure>: <summarized observation> <action>"**

In [ ]:
# First, recall, get, say, 10 memories
obs = trajectory_data[2]
summarized_obs = mm.summarize_webarena_observation(obs["observation"]["text"])
mems = mm.cue_based_recall(summarized_obs=summarized_obs,
    goal=task_from_trajectory,
    top_k=10)

In [ ]:
formatted_mems = []
for m in mems:
    # If there is an 'obs_summary' field, then we know it's a cue-action mapping
    if 'obs_summary' in m:
        success = "success" if m['success'] else "failure"
        pointer = "(DONT DO AGAIN)" if m['success'] else ""
        formatted_mems.append(f"""Last time I was in a similar situation, I tried doing the corresponding action, and it ultimately led to {success}:

WHAT I SAW:
{m['obs_summary']}

WHAT I DID{pointer}:
{m['action_taken']}
""")
    else: # Learned skills -> just take the embedding (TODO)
        ...

In [67]:
print(formatted_mems[0])

Last time I was in a similar situation, I tried doing the corresponding action, and it ultimately led to failure:
WHAT I SAW:
**Summary of Web Observation – Magento Admin Dashboard:**

The current page is the **Magento Admin Dashboard** at `http://ec2-3-128-25-196.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/`. Key observations include:

- **Navigation**: The top horizontal menu provides access to core Magento sections (Sales, Catalog, Customers, Marketing, Stores, System, etc.), with “Dashboard” active.
- **System Alerts**: 
  - **2 system messages** are present.
  - **Cache Warning**: The “Configuration” cache type is invalidated; users are prompted to visit **Cache Management** to refresh it.
- **Dashboard Overview**:
  - **Scope**: Set to “All Store Views”; option to reload data is available.
  - **Advanced Reporting**: A call-to-action links to advanced analytics, though the chart is currently disabled (user must click a link to enable it).
  - **Sales

WHAT I DID:
c

Now, we can have our recall agent take a list of k top memories (formatted above), give us back x memories it believes will lead to maximum reward (quicker and correct task completion), where x is dictated by its entropy. Then, create a new prompt template in webarena, which is cot, but now with a slot for inserting memories spit out by our recall agent.

## Entropy

In [43]:
from typing import List
from litellm import completion

In [ ]:
hardcoded_full_prompt = """
You are an autonomous intelligent agent tasked with navigating a web browser. You will be given web-based tasks. These tasks will be accomplished through the use of specific actions you can issue.

Here's the information you'll have:
The user's objective: This is the task you're trying to complete.
The current web page's accessibility tree: This is a simplified representation of the webpage, providing key information.
The current web page's URL: This is the page you're currently navigating.
The open tabs: These are the tabs you have open.
The previous action: This is the action you just performed. It may be helpful to track your progress.

The actions you can perform fall into several categories:

Page Operation Actions:
`click [id]`: This action clicks on an element with a specific id on the webpage.
`type [id] [content] [press_enter_after=0|1]`: Use this to type the content into the field with id. By default, the "Enter" key is pressed after typing unless press_enter_after is set to 0.
`hover [id]`: Hover over an element with id.
`press [key_comb]`:  Simulates the pressing of a key combination on the keyboard (e.g., Ctrl+v).
`scroll [direction=down|up]`: Scroll the page up or down.

Tab Management Actions:
`new_tab`: Open a new, empty browser tab.
`tab_focus [tab_index]`: Switch the browser's focus to a specific tab using its index.
`close_tab`: Close the currently active tab.

URL Navigation Actions:
`goto [url]`: Navigate to a specific URL.
`go_back`: Navigate to the previously viewed page.
`go_forward`: Navigate to the next page (if a previous 'go_back' action was performed).

Completion Action:
`stop [answer]`: Issue this action when you believe the task is complete. If the objective is to find a text-based answer, provide the answer in the bracket.

Homepage:
If you want to visit other websites, check out the homepage at http://homepage.com. It has a list of websites you can visit.
http://homepage.com/password.html lists all the account name and password for the websites. You can use them to log in to the websites.

To be successful, it is very important to follow the following rules:
1. You should only issue an action that is valid given the current observation
2. You should only issue one action at a time.
4. Generate the action in the correct format, wrap the action inside ``````. For example, ```click [1234]```".
5. Issue stop action when you think you have achieved the objective.

Here are a few examples:
OBSERVATION:
[1744] link 'HP CB782A#ABA 640 Inkjet Fax Machine (Renewed)'
		[1749] StaticText '$279.49'
		[1757] button 'Add to Cart'
		[1760] button 'Add to Wish List'
		[1761] button 'Add to Compare'
URL: http://onestopmarket.com/office-products/office-electronics.html
OBJECTIVE: What is the price of HP Inkjet Fax Machine
PREVIOUS ACTION: None
Action: ```stop [$279.49]```

OBSERVATION:
[164] textbox 'Search' focused: True required: False
[171] button 'Go'
[174] link 'Find directions between two points'
[212] heading 'Search Results'
[216] button 'Close'
URL: http://openstreetmap.org
OBJECTIVE: Show me the restaurants near CMU
PREVIOUS ACTION: None
Action: ```type [164] [restaurants near CMU] [1]```

Now make prediction given the observation:

OBSERVATION:
Tab 0 (current): Dashboard / Magento Admin

[251] RootWebArea 'Dashboard / Magento Admin' focused: True url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/
	[547] link 'Magento Admin Panel' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/
		[548] image 'Magento Admin Panel' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/static/version1681922233/adminhtml/Magento/backend/en_US/images/magento-icon.svg
	[550] menubar '' orientation: horizontal
		[552] link '\ue604 DASHBOARD' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/
		[555] link '\ue60b SALES' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/#
		[588] link '\ue608 CATALOG' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/#
		[606] link '\ue603 CUSTOMERS' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/#
		[625] link '\ue609 MARKETING' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/#
		[692] link '\ue602 CONTENT' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/#
		[738] link '\ue60a REPORTS' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/#
		[863] link '\ue60d STORES' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/#
		[939] link '\ue610 SYSTEM' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/#
		[1022] link '\ue612 FIND PARTNERS & EXTENSIONS' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/marketplace/index/
	[1412] button 'System Messages: 2'
	[1574] StaticText 'One or more of the Cache Types are invalidated: Configuration. Please go to '
	[1573] link 'Cache Management' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/cache/
	[1576] StaticText ' and refresh cache types.'
	[1044] heading 'Dashboard'
	[1047] link '\ue600 admin' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/system_account/index/
	[256] link '\ue607' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/notification/index/
	[258] textbox '\ue60c' required: False
	[1066] main ''
		[157] StaticText 'Scope:'
		[1077] button 'All Store Views' hasPopup: menu
		[1088] link '\ue633 What is this?' url: https://docs.magento.com/user-guide/configuration/scope.html
		[1095] button 'Reload Data'
		[1100] sectionheader ''
			[164] StaticText 'Advanced Reporting'
		[165] StaticText "Gain new insights and take command of your business' performance, using our dynamic product, order, and customer reports tailored to your customer data."
		[1103] link 'Go to Advanced Reporting \ue644' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/analytics/reports/show/
		[169] StaticText 'Chart is disabled. To enable the chart, click '
		[1109] link 'here' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/system_config/edit/section/admin/#admin_dashboard-link
		[172] StaticText 'Revenue'
		[173] StaticText '$0.00'
		[174] StaticText 'Tax'
		[176] StaticText 'Shipping'
		[178] StaticText 'Quantity'
		[179] StaticText '0'
		[1134] tablist '' multiselectable: False orientation: horizontal
			[314] tab 'The information in this tab has been changed. This tab contains invalid data. Please resolve this before saving. Loading... Bestsellers' expanded: True selected: True controls: grid_tab_ordered_products_content
				[1135] link 'Bestsellers' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/#grid_tab_ordered_products_content
			[317] tab 'The information in this tab has been changed. This tab contains invalid data. Please resolve this before saving. Loading... Most Viewed Products' expanded: False selected: False
				[1141] link 'Most Viewed Products' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/productsViewed/
			[319] tab 'The information in this tab has been changed. This tab contains invalid data. Please resolve this before saving. Loading... New Customers' expanded: False selected: False
				[1148] link 'New Customers' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/customersNewest/
			[321] tab 'The information in this tab has been changed. This tab contains invalid data. Please resolve this before saving. Loading... Customers' expanded: False selected: False
				[1155] link 'Customers' url: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/customersMost/
		[315] tabpanel 'The information in this tab has been changed. This tab contains invalid data. Please resolve this before saving. Loading... Bestsellers'
			[329] table ''
				[331] rowgroup ''
					[333] row ''
						[335] columnheader 'Product' required: False
						[339] columnheader 'Price' required: False
						[343] columnheader 'Quantity' required: False
				[351] row 'http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/catalog/product/edit/id/29/'
					[353] cell 'Sprite Stasis Ball 65 cm'
					[356] cell '$27.00'
					[359] cell '6'
		[180] StaticText 'Lifetime Sales'
		[181] StaticText '$0.00'
		[182] StaticText 'Average Order'
		[184] StaticText 'Last Orders'
		[1180] table ''
			[1181] rowgroup ''
				[1182] row ''
					[1183] columnheader 'Customer' required: False
					[1185] columnheader 'Items' required: False
					[1187] columnheader 'Total' required: False
			[1190] row 'http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/sales/order/view/order_id/299/'
				[1191] cell 'Sarah Miller'
				[1192] cell '5'
				[1193] cell '$194.40'
			[1194] row 'http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/sales/order/view/order_id/65/'
				[1195] cell 'Grace Nguyen'
				[1196] cell '4'
				[1197] cell '$190.00'
URL: http://ec2-18-224-1-226.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/
OBJECTIVE: What is the top-1 best-selling brand in Quarter 1 2022
PREVIOUS ACTION: None

Action:
"""

# Getting comfortable with the lobprob outputs
def generate_from_litellm_completion(
    prompt: str,
    model: str,
    temperature: float = 0.0,
    max_tokens: int = 4096,
    system_prompt: str = None,
    stop_sequences: List[str] | None = None,
):
    if not os.getenv("TOGETHER_API_KEY"):
        raise ValueError("TOGETHER_API_KEY environment variable must be set.")
    
    messages = [{"content": prompt, "role": "user"}]
    if system_prompt:
        messages.insert(0, {"content": system_prompt, "role": "system"})
    
    # Call completion directly like the working LiteLLMModel does
    try:
        response = completion(
            model=model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
            stop=stop_sequences,
            logprobs=True,
			top_logprobs=5
        )
    except Exception as e:
        print(f"Error: {e}")
        return None

    # Together AI returns text directly in choices[0].text
    #return response["choices"][0]["message"]["content"]
    return response

response_data = generate_from_litellm_completion(
    prompt=hardcoded_full_prompt,
    model="together_ai/OpenAI/gpt-oss-120B",
    temperature=0.7,
)


In [39]:
response_data

In [ ]:
# Fixed version of the logprobs analysis
def format_top_alternatives_fixed(token_data, max_alternatives=5):
    """Format the top alternatives for a token - FIXED VERSION"""
    if 'top_logprobs' not in token_data or not token_data['top_logprobs']:
        return "No alternatives"
    
    alternatives = []
    for i, alt in enumerate(token_data['top_logprobs'][:max_alternatives]):
        marker = "✓" if i == 0 else " "
        # Access attributes directly since alt is a TopLogprob object
        alternatives.append(f"{marker} {alt.token} ({alt.logprob:.3f})")
    return " | ".join(alternatives)

def calculate_confidence_gap_fixed(token_data):
    """Calculate the gap between 1st and 2nd choice - FIXED VERSION"""
    if 'top_logprobs' not in token_data or len(token_data['top_logprobs']) < 2:
        return None
    return token_data['top_logprobs'][0].logprob - token_data['top_logprobs'][1].logprob

# Extract logprobs data
logprobs_data = response_data['choices'][0]['logprobs']['content']

# Create a detailed DataFrame with fixed functions
detailed_df = pd.DataFrame([
    {
        'token': token['token'],
        'logprob': token['logprob'],
        'alternatives': format_top_alternatives_fixed(token),
        'confidence_gap': calculate_confidence_gap_fixed(token),
        'bytes': str(token['bytes'])[:20] + '...' if len(str(token['bytes'])) > 20 else str(token['bytes'])
    }
    for token in logprobs_data
])

print("=== TOKENS WITH TOP 5 ALTERNATIVES ===")
print(detailed_df.to_string(index=False, max_colwidth=80))
print(f"\n... and {len(logprobs_data) - 10} more tokens")

# Show statistics
logprobs_values = [token['logprob'] for token in logprobs_data]
confidence_gaps = [calculate_confidence_gap_fixed(token) for token in logprobs_data if calculate_confidence_gap_fixed(token) is not None]

print(f"\n=== LOGPROB STATISTICS ===")
print(f"Total tokens: {len(logprobs_data)}")
print(f"Min logprob: {min(logprobs_values):.6f}")
print(f"Max logprob: {max(logprobs_values):.6f}")
print(f"Mean logprob: {sum(logprobs_values)/len(logprobs_values):.6f}")

if confidence_gaps:
    print(f"\n=== CONFIDENCE GAP STATISTICS ===")
    print(f"Min confidence gap: {min(confidence_gaps):.6f}")
    print(f"Max confidence gap: {max(confidence_gaps):.6f}")
    print(f"Mean confidence gap: {sum(confidence_gaps)/len(confidence_gaps):.6f}")

# Show most confident tokens
high_confidence = sorted(logprobs_data, key=lambda x: x['logprob'], reverse=True)[:5]
print(f"\n=== TOP 5 MOST CONFIDENT TOKENS ===")
for token in high_confidence:
    print(f"'{token['token']}' (logprob: {token['logprob']:.6f})")

# Show most uncertain tokens (smallest confidence gaps)
uncertain_tokens = [(token, calculate_confidence_gap_fixed(token)) for token in logprobs_data if calculate_confidence_gap_fixed(token) is not None]
uncertain_tokens.sort(key=lambda x: x[1])  # Sort by confidence gap (ascending)

print(f"\n=== TOP 5 MOST UNCERTAIN TOKENS (smallest confidence gaps) ===")
for token, gap in uncertain_tokens[:5]:
    first_choice = token['top_logprobs'][0].token
    second_choice = token['top_logprobs'][1].token
    print(f"'{token['token']}' -> gap: {gap:.3f} (1st: '{first_choice}', 2nd: '{second_choice}')")

In [ ]:
# Compute entropy per token and average entropy
import numpy as np

def entropy_from_top_logprobs(top_logprobs):
    """
    top_logprobs: list[TopLogprob] for a single generated token.
    """
    logps = np.array([t.logprob for t in top_logprobs])
    # Convert logprobs → probs safely
    probs = np.exp(logps - np.max(logps))  # numerical stability
    probs /= probs.sum()
    # Compute entropy (nats)
    return -np.sum(probs * np.log(probs + 1e-9))


def get_mean_and_action_entropies(logprobs_data):
    action_decision_entropy = None
    entropies = []

    # TODO: Confirm that tokens indeed look like the following actions when we're on the action generation (e.g. go_back is not 2 tokens)
    actions = [
        'click',
        'type',
        'hover',
        'press',
        'scroll',
        'new_tab',
        'tab_focus',
        'close_tab',
        'goto',
        'go_back',
        'go_forward',
        'stop'
    ]

    for token_info in logprobs_data:  # your list of ChatCompletionTokenLogprob
        h = entropy_from_top_logprobs(token_info.top_logprobs)
        entropies.append(h)

        if token_info.token in actions:
            action_decision_entropy = h

    mean_entropy = np.mean(entropies)

    return mean_entropy, action_decision_entropy

mean_entropy, action_decision_entropy = get_mean_and_action_entropies(logprobs_data)
print(f"Mean entropy: {mean_entropy:.4f} | Action decision entropy: {action_decision_entropy:.4f}")

Now to collect mean entropies, for all trajectory steps (assuming the source code is changed with above code, to record entropy per env step, for litellm generations)

In [60]:
all_mean_entropies = []

In [12]:
# Import necessary modules for manual agent action generation
from webarena.agent.agent import PromptAgent
from webarena.llms import lm_config
from webarena.agent.prompts import PromptConstructor
from webarena.browser_env.actions import create_id_based_action, create_playwright_action, create_none_action
from webarena.browser_env.helper_functions import get_action_description
from webarena.llms import generate_from_litellm_completion
import re


# Initialize the agent with the same configuration as used in the original run
agent = PromptAgent(
    action_set_tag="id_accessibility_tree",
    model="together_ai/OpenAI/gpt-oss-120B",
    temperature=0.7,
    use_litellm=True,
    instruction_path="../webarena/agent/prompts/raw/p_direct_id_actree_2s_no_na.py",
    verbose=False # set to True if we want to see the full prompt and agent response under the hood each time
)

# Function to manually generate action for a given observation
def manual_action_generation(trajectory_item, intent, previous_action="None"):
    """
    Manually call the agent's action generation for a single trajectory observation
    """
    # Extract observation details
    observation = trajectory_item["observation"]["text"]
    url = trajectory_item["info"]["page"].url
    
    # Create a minimal trajectory with just this observation
    single_observation_trajectory = [trajectory_item]
    
    # Create metadata with action history
    meta_data = {"action_history": [previous_action]}
    
    # Call the agent's action generation method
    try:
        return generate_from_litellm_completion(
            prompt=prompt,
            model="together_ai/OpenAI/gpt-oss-120B",
            temperature=0.7,
        )
    except Exception as e:
        print(f"Error generating action: {e}")
        return None


In [62]:
first_observation = trajectory_data[0]

print("=== Generating Action for First Observation ===")
print(f"URL: {first_observation['info']['page'].url}")
print(f"Task: {task_from_trajectory}")
print("\nObservation text (first 500 chars):")
print(first_observation['observation']['text'][:500] + "...")

# Generate action for this observation
response = manual_action_generation(
    trajectory_item=first_observation,
    intent=task_from_trajectory,
    previous_action="None"
)

if response:
    action = response["action"]
    mean_entropy = response["mean_entropy"]
    action_decision_entropy = response["action_decision_entropy"]
    print(f"\nGenerated Action:")
    print(f"Action type: {action.get('action_type', 'unknown')}")
    print(f"Element ID: {action.get('element_id', 'none')}")
    print(f"Raw prediction: {action.get('raw_prediction', 'none')}")
    print(f"LLM reasoning: {action.get('llm_reasoning', 'none')}")
    print(f"  Mean entropy: {mean_entropy:.4f} | Action decision entropy: {action_decision_entropy:.4f}")
else:
    print("Failed to generate action")


=== Generating Action for First Observation ===
URL: http://ec2-3-128-25-196.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/
Task: What is the top-1 best-selling brand in Quarter 1 2022

Observation text (first 500 chars):
Tab 0 (current): Dashboard / Magento Admin

[251] RootWebArea 'Dashboard / Magento Admin' focused: True url: http://ec2-3-128-25-196.us-east-2.compute.amazonaws.com:7780/admin/admin/dashboard/
	[547] link 'Magento Admin Panel' url: http://ec2-3-128-25-196.us-east-2.compute.amazonaws.com:7780/admin/admin/
		[548] image 'Magento Admin Panel' url: http://ec2-3-128-25-196.us-east-2.compute.amazonaws.com:7780/static/version1681922233/adminhtml/Magento/backend/en_US/images/magento-icon.svg
	[550] menu...
Error generating action: name 'prompt' is not defined
Failed to generate action


In [ ]:
#  Process all observations in the trajectory and generate actions for each
generated_actions = []
traj_so_far = []
task_intent = task_from_trajectory

print(f"Processing {len(trajectory_data)} observations...")
print("=" * 80)

meta_data = {"action_history": ["None"]}

for i in range(0, len(trajectory_data), 2):
    observation_item = trajectory_data[i]
    traj_so_far.append(observation_item)
    
    print(f"\n--- Processing Observation {i+1} ---")
    print(f"URL: {observation_item['info']['page'].url}")
    
    # Get previous action if available
    previous_action = "None"
    if i > 0 and len(generated_actions) > 0:
        prev_action = generated_actions[-1]
        if prev_action and 'raw_prediction' in prev_action:
            previous_action = prev_action['raw_prediction']
    
    response = agent._next_action_litellm(
        trajectory=traj_so_far,
        intent=task_intent,
        meta_data=meta_data
    )

    if response:
        action = response["action"]
        mean_entropy = response["mean_entropy"]
        all_mean_entropies.append(mean_entropy)
        action_decision_entropy = response["action_decision_entropy"]
        generated_actions.append(action)
        print(f"✓ Generated action: {action.get('action_type', 'unknown')}")
        print(f"  Element ID: {action.get('element_id', 'none')}")
        print(f"  Reasoning: {action.get('llm_reasoning', 'none')[:100]}...")
        print(f"  Mean entropy: {mean_entropy:.4f} | Action decision entropy: {action_decision_entropy:.4f}")
    else:
        print("✗ Failed to generate action")
        generated_actions.append(None)
    
    action_str = get_action_description(
        action,
        observation_item["info"]["observation_metadata"],
        action_set_tag="id_accessibility_tree",
        prompt_constructor=agent.prompt_constructor
        if isinstance(agent, PromptAgent)
        else None,
    )
    meta_data["action_history"].append(action_str)

    traj_so_far.append(action)
    
print(f"\n=== Summary ===")
print(f"Successfully generated {sum(1 for a in all_actions if a is not None)} actions out of {len(all_actions)} observations")
